## 01. 회귀 평가지표  

회귀 평가지표는 모델이 예측한 숫자와 실제 숫자의 차이를 계산해 회귀 모델의 성능을 수치로 표현한 기준이다.  
회귀 모델은 class가 아니라 연속적인 숫자 값을 예측하므로, 평가지표는 예측값이 실제값과 얼마나 가까운지 확인하는 데 사용한다.  

회귀 평가는 단순히 맞음/틀림으로 판단하지 않는다.  
예측값과 실제값 사이의 차이가 얼마나 큰지, 큰 오차를 얼마나 민감하게 볼 것인지, 평균값만 예측하는 기준보다 얼마나 나은지를 함께 확인한다.  

- `MAE`: 오차의 절댓값을 평균낸 지표임.  
- `MSE`: 오차를 제곱한 뒤 평균낸 지표임.  
- `RMSE`: MSE에 제곱근을 씌워 실제 target과 비슷한 단위로 해석하는 지표임.  
- `R2`: 평균값으로만 예측하는 기준보다 모델이 target 변화를 얼마나 잘 설명하는지 보는 지표임.  

**회귀 모델과 평가의 관계**  

- 회귀 모델: 숫자 값을 예측하는 모델임.  
- 회귀 평가지표: 모델의 숫자 예측값이 실제값과 얼마나 가까운지 판단하는 기준임.  

분류에서는 맞음/틀림을 기준으로 accuracy를 계산할 수 있지만, 회귀에서는 예측값이 실제값과 얼마나 차이 나는지가 중요하다.  
예측값이 100이고 실제값이 102라면 완전히 틀렸다고만 볼 수 없다. 오차가 2 정도라면 충분히 좋은 예측일 수 있다.  

**배우는 이유**  
- 회귀 모델의 예측 오차 크기를 숫자로 판단하기 위해 사용함.  
- 큰 오차가 치명적인 문제인지, 평균적인 오차가 중요한 문제인지 구분하기 위해 사용함.  
- 여러 회귀 모델 중 어떤 모델을 선택할지 근거를 만들기 위해 사용함.  
- 프로젝트 보고서에서 “모델이 좋다”가 아니라 “평균적으로 얼마나 틀리는지”를 설명하기 위해 사용함.  

**어디서 사용하는가?**  
- 집값 예측: 평균적으로 몇 천만 원 정도 틀리는지 확인함.  
- 수요 예측: 재고 부족이나 과잉 재고를 줄이기 위해 오차를 확인함.  
- 매출 예측: 실제 매출과 예측 매출의 차이를 기준으로 모델을 비교함.  
- 의료 수치 예측: 큰 오차가 위험할 수 있어 RMSE를 함께 확인함.  

**핵심 용어**  
- 실제값: 정답 target 값임.  
- 예측값: 모델이 만든 숫자 결과임.  
- 잔차(residual): 실제값 - 예측값임.  
- 오차(error): 실제값과 예측값의 차이를 의미함.  
- 기준 모델: 복잡한 모델을 만들기 전에 최소한 이겨야 하는 단순 예측 기준임.  

**지표를 읽는 기본 방향**  
- MAE, MSE, RMSE는 낮을수록 예측 오차가 작다.  
- R2는 일반적으로 높을수록 평균 예측보다 모델이 target 변화를 더 잘 설명한다.  
- 단, R2만 높다고 좋은 모델이라고 단정하지 않고 MAE/RMSE의 실제 오차 크기를 함께 확인한다.  


## 02. 실습 환경 준비

회귀 평가지표는 직접 계산해 보면 의미가 훨씬 분명해진다.
작은 배열 예제로 지표를 먼저 확인한 뒤, Diabetes 데이터로 실제 모델 평가를 진행한다.


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes

from sklearn.model_selection import train_test_split

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
)


## 03. 작은 예제로 residual 이해하기

residual은 실제값에서 예측값을 뺀 값이다.
residual이 양수이면 모델이 실제보다 작게 예측한 것이고, 음수이면 실제보다 크게 예측한 것이다.


In [ ]:
y_true = np.array([3.0, 0.5, 2.0, 7.0])
y_pred = np.array([2.5, 0.0, 2.0, 5.0])

# residual: 실제값 - 예측값
# 양수이면 모델이 실제값보다 작게 예측한 것이고, 음수이면 실제값보다 크게 예측한 것이다.
residual = y_true - y_pred

residual_df = pd.DataFrame({
    'actual': y_true,
    'predicted': y_pred,
    'residual_actual_minus_pred': residual,
    'absolute_error': np.abs(residual),
    'squared_error': residual ** 2,
})

display(residual_df)

## 04. MAE, MSE, RMSE, R2 직접 계산

- MAE: 오차 절댓값 평균임.
- MSE: 오차 제곱 평균임.
- RMSE: MSE에 제곱근을 씌운 값임.
- R2: 평균값으로 예측하는 기준보다 모델이 얼마나 더 잘 설명하는지 보는 지표임.


## 05. Diabetes 회귀 데이터 로드

Diabetes 데이터는 여러 수치 feature를 사용해 질병 진행 정도를 예측하는 회귀 데이터셋이다.
target은 연속적인 숫자이므로 분류 지표가 아니라 회귀 지표로 평가한다.


In [ ]:
diabetes = load_diabetes(as_frame=True)

diabetes_X = diabetes.data
diabetes_y = diabetes.target

print('feature shape:', diabetes_X.shape)
print('target shape:', diabetes_y.shape)
display(diabetes_X.head())
display(diabetes_y.describe().to_frame('target_summary'))


## 06. 학습/평가 데이터 분리

회귀 모델도 분류 모델과 마찬가지로 train/test를 분리한다.
최종 평가지표는 학습에 사용하지 않은 test 데이터에서 계산해야 한다.


In [ ]:
diabetes_X_train, diabetes_X_test, diabetes_y_train, diabetes_y_test = train_test_split(
    diabetes_X,
    diabetes_y,
    test_size=0.2,
    random_state=42,
)

print('train:', diabetes_X_train.shape, diabetes_y_train.shape)
print('test:', diabetes_X_test.shape, diabetes_y_test.shape)


## 07. 비교할 회귀 모델 구성

회귀 지표는 여러 모델을 비교할 때 더 유용하다.
여기서는 평균 예측 기준 모델, 선형 회귀, Ridge, RandomForestRegressor를 비교한다.


## 08. 회귀 모델별 지표 비교

MAE, MSE, RMSE는 낮을수록 좋고, R2는 일반적으로 높을수록 좋다.
단, R2가 높아도 오차 크기가 업무적으로 허용 가능한지는 별도로 판단해야 한다.


## 09. 실제값과 예측값 비교 그래프

회귀 모델은 예측값이 실제값과 가까울수록 좋다.
산점도가 대각선 근처에 모이면 예측이 실제값과 비슷하다는 뜻이다.


## 10. 잔차 분포 확인

잔차는 실제값에서 예측값을 뺀 값이다.
잔차가 0 근처에 모이면 예측 오차가 작다는 뜻이고, 한쪽으로 치우치면 과대평가나 과소평가 경향이 있을 수 있다.


## 11. 큰 오차에 민감한 지표 비교

MAE는 오차를 그대로 평균내므로 직관적이다.
MSE와 RMSE는 오차를 제곱하기 때문에 큰 오차가 하나만 있어도 지표가 크게 나빠질 수 있다.


## 12. 모델 평가 리포트 문장 만들기

평가지표는 숫자만 출력하고 끝내면 수업이나 프로젝트 보고서에서 의미가 부족하다.
어떤 모델을 선택했고, 그 이유가 무엇인지 문장으로 정리해야 한다.


## 13. 회귀 평가지표 정리

- Residual: 실제값과 예측값의 차이임.
- MAE: 오차 절댓값 평균임. 실제 단위로 설명하기 쉬움.
- MSE: 오차 제곱 평균임. 큰 오차에 민감함.
- RMSE: MSE에 제곱근을 씌운 값임. target과 비슷한 단위로 해석함.
- R2: 평균 예측보다 모델이 얼마나 더 잘 설명하는지 보는 지표임.

회귀 지표는 낮을수록 좋은 지표와 높을수록 좋은 지표가 섞여 있다.
MAE, MSE, RMSE는 낮을수록 좋고, R2는 일반적으로 높을수록 좋다.
